# LeetCode #137 — Single Number II: đếm bit modulo 3

**Mức độ:** Medium · **Chủ đề:** Array, Bit Manipulation

Skeleton trên LeetCode:

```python
class Solution:
    def singleNumber(self, nums: List[int]) -> int:
```

Mục tiêu của notebook:

1. Nhớ lại #136: XOR khử các cặp — nhưng ở đây mỗi số lặp **ba** lần, XOR không còn tác dụng.
2. Ý tưởng chính: **đếm từng bit** qua mọi số, lấy **modulo 3**.
3. Xử lý **số âm** trong Python (số nguyên không giới hạn bit) — chuyển từ bù hai 32-bit.
4. Phiên bản "máy trạng thái" `ones` / `twos` — `O(1)` bộ nhớ, một lượt.

# 1. Bài toán là gì?

Cho mảng `nums`, **mọi** phần tử xuất hiện **đúng ba lần**, trừ **một** phần tử xuất hiện **đúng một lần**. Tìm phần tử đó.

Yêu cầu: thời gian **tuyến tính**, bộ nhớ phụ **hằng số**.

**Constraints (theo đề):**

- `1 <= nums.length <= 3 * 10^4`
- `-2^31 <= nums[i] <= 2^31 - 1`
- Mọi phần tử xuất hiện ba lần, trừ đúng một phần tử xuất hiện một lần.

---

## Ví dụ

```text
[2, 2, 3, 2]                  ->  3
[0, 1, 0, 1, 0, 1, 99]        ->  99
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Biểu diễn nhị phân và các phép toán bit

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Biểu diễn nhị phân và các phép toán bit

### Số nguyên dưới dạng bit

Máy tính lưu số nguyên ở hệ **nhị phân**: mỗi chữ số (bit) là 0 hoặc 1, bit thứ `k` (đếm từ 0, bên phải) có giá trị `2^k`.

```text
13 = 8 + 4 + 1 = 1·2³ + 1·2² + 0·2¹ + 1·2⁰ = 0b1101
bit:              3     2      1      0
```

In [1]:
x_bit = 13
print("bin(13) =", bin(x_bit), "| format 8 bit:", format(x_bit, "08b"), "| int('1101', 2) =", int("1101", 2))
print("bit thứ k của 13:", [(x_bit >> k) & 1 for k in range(4)], "(từ bit 0)")

bin(13) = 0b1101 | format 8 bit: 00001101 | int('1101', 2) = 13
bit thứ k của 13: [1, 0, 1, 1] (từ bit 0)


### Sáu phép toán bit

| Phép | Ký hiệu | Quy tắc từng bit | Ví dụ `12 = 1100`, `10 = 1010` |
|---|---|---|---|
| AND | `a & b` | 1 nếu **cả hai** là 1 | `1000` = 8 |
| OR | `a \| b` | 1 nếu **ít nhất một** là 1 | `1110` = 14 |
| XOR | `a ^ b` | 1 nếu **khác nhau** | `0110` = 6 |
| NOT | `~a` | đảo mọi bit | `~12 = -13` (Python) |
| dịch trái | `a << k` | thêm `k` số 0 bên phải = nhân `2^k` | `12 << 1 = 24` |
| dịch phải | `a >> k` | bỏ `k` bit phải = chia nguyên `2^k` | `12 >> 2 = 3` |

In [2]:
a_b, b_b = 12, 10
for ten, v in [("a & b", a_b & b_b), ("a | b", a_b | b_b), ("a ^ b", a_b ^ b_b), ("a << 1", a_b << 1), ("a >> 2", a_b >> 2), ("~a", ~a_b)]:
    print(f"{ten:<7} = {v:>4}   bin: {bin(v)}")

a & b   =    8   bin: 0b1000
a | b   =   14   bin: 0b1110
a ^ b   =    6   bin: 0b110
a << 1  =   24   bin: 0b11000
a >> 2  =    3   bin: 0b11
~a      =  -13   bin: -0b1101


### Các mẹo hay dùng

| Mục đích | Biểu thức |
|---|---|
| Kiểm tra bit `k` | `(x >> k) & 1` |
| Bật / tắt / lật bit `k` | `x \| (1 << k)`, `x & ~(1 << k)`, `x ^ (1 << k)` |
| Số chẵn? | `x & 1 == 0` |
| Bỏ bit 1 thấp nhất | `x & (x - 1)` |
| Lấy bit 1 thấp nhất | `x & -x` |
| Là lũy thừa của 2? | `x > 0 and x & (x - 1) == 0` |

**Tính chất XOR** (cực kỳ hay dùng): `x ^ x = 0`, `x ^ 0 = x`, giao hoán và kết hợp → XOR cả dãy thì các số xuất hiện **chẵn** lần triệt tiêu nhau.

In [3]:
x_m = 0b10110
print("bỏ bit 1 thấp nhất:", format(x_m, "05b"), "→", format(x_m & (x_m - 1), "05b"))
print("lấy bit 1 thấp nhất:", format(x_m & -x_m, "05b"))
print("lũy thừa của 2:", [v for v in range(1, 40) if v & (v - 1) == 0])
print("đếm bit 1 của 255:", bin(255).count("1"))
xor_all = 0
for v in [4, 1, 2, 1, 2]:
    xor_all ^= v
print("XOR [4,1,2,1,2] =", xor_all, "(số xuất hiện lẻ lần)")

bỏ bit 1 thấp nhất: 10110 → 10100
lấy bit 1 thấp nhất: 00010
lũy thừa của 2: [1, 2, 4, 8, 16, 32]
đếm bit 1 của 255: 8
XOR [4,1,2,1,2] = 4 (số xuất hiện lẻ lần)


### Dùng bit làm tập hợp (bitmask)

Với `n` phần tử, một số `mask` từ `0` tới `2^n − 1` biểu diễn một **tập con**: bit `i` bằng 1 nghĩa là lấy phần tử `i`. Duyệt mọi `mask` = duyệt mọi tập con.

In [4]:
items = ["a", "b", "c"]
for mask in range(1 << len(items)):
    tap = [items[i] for i in range(len(items)) if mask >> i & 1]
    print(format(mask, "03b"), tap)

000 []
001 ['a']
010 ['b']
011 ['a', 'b']
100 ['c']
101 ['a', 'c']
110 ['b', 'c']
111 ['a', 'b', 'c']


### Số âm và giới hạn 32 bit

Ngôn ngữ như C/Java lưu `int` bằng 32 bit, số âm dạng **bù hai** (two's complement): `-x = ~x + 1`. Python có số nguyên **không giới hạn**, nên muốn mô phỏng 32 bit phải **che** bằng `& 0xFFFFFFFF` rồi đổi lại thành số có dấu.

In [5]:
def thanh_32bit(x):
    x &= 0xFFFFFFFF
    return x - (1 << 32) if x >> 31 else x       # bit 31 là bit dấu

print("-1 ở 32 bit:", format(-1 & 0xFFFFFFFF, "032b"))
print("2^31 tràn thành:", thanh_32bit(2 ** 31), "| (2^31 - 1) + 1 =", thanh_32bit(2 ** 31 - 1 + 1))

-1 ở 32 bit: 11111111111111111111111111111111
2^31 tràn thành: -2147483648 | (2^31 - 1) + 1 = -2147483648


Lỗi hay gặp: nhầm `^` là lũy thừa (lũy thừa là `**`); quên ngoặc vì `&`, `|`, `^` có độ ưu tiên **thấp hơn** `==` — viết `(x & 1) == 0`; `~x` trong Python là `-x - 1`, không phải "đảo 32 bit".

## 2.2 Áp dụng vào bài này

Mọi số xuất hiện 3 lần trừ một số. **Đếm bit**: với mỗi vị trí bit, tổng các bit `% 3` chính là bit của số cần tìm (các số lặp 3 lần đóng góp bội của 3). Cần xử lý **số âm** kiểu 32 bit. Cách gọn hơn dùng hai biến `ones`, `twos` và các phép `^`, `&`, `~`.

# 3. Baseline: đếm bằng hash map

`O(n)` thời gian nhưng `O(n)` bộ nhớ — vi phạm yêu cầu.

In [6]:
from typing import List
from collections import Counter
from functools import reduce
from operator import xor
import random

def single_counter(nums: List[int]) -> int:
    return next(x for x, c in Counter(nums).items() if c == 1)

print(single_counter([2, 2, 3, 2]), single_counter([0, 1, 0, 1, 0, 1, 99]))

3 99


Một mẹo toán học khác: `(3 · sum(set(nums)) − sum(nums)) / 2` — vẫn cần `set` (`O(n)` bộ nhớ).

In [7]:
def single_math(nums: List[int]) -> int:
    return (3 * sum(set(nums)) - sum(nums)) // 2

print(single_math([2, 2, 3, 2]), single_math([0, 1, 0, 1, 0, 1, 99]))

3 99


# 4. Vì sao XOR (cách của #136) không dùng được?

`a ^ a = 0` khử **cặp**. Với ba bản: `a ^ a ^ a = a` — không khử được.

In [8]:
print("XOR toàn bộ [2, 2, 3, 2] =", reduce(xor, [2, 2, 3, 2]), "(không phải 3)")

XOR toàn bộ [2, 2, 3, 2] = 1 (không phải 3)


# 5. Ý tưởng chính: đếm bit modulo 3

Xét **từng vị trí bit** `b` (0..31). Đếm số phần tử có bit `b` bật:

- Mỗi số xuất hiện ba lần đóng góp `0` hoặc `3` vào bộ đếm.
- Số duy nhất đóng góp `0` hoặc `1`.

⇒ `count_b % 3` chính là bit `b` của đáp án.

```text
nums = [2, 2, 3, 2]   (nhị phân: 10, 10, 11, 10)
bit 0:  0 + 0 + 1 + 0 = 1  -> 1 % 3 = 1
bit 1:  1 + 1 + 1 + 1 = 4  -> 4 % 3 = 1
đáp án = 11 (nhị phân) = 3
```

## 5.1 Số âm trong Python

Số nguyên Python **không giới hạn bit**: `-1 >> 31` vẫn là `-1`, và `-1 & (1 << 31)` khác 0. Ta xét đúng **32 bit thấp** (bù hai), rồi nếu bit 31 bật thì đổi lại thành số âm: `x − 2^32`.

In [9]:
def to_signed32(x: int) -> int:
    x &= 0xFFFFFFFF
    return x - (1 << 32) if x & (1 << 31) else x

for v in [5, -5, -1, -2**31, 2**31 - 1]:
    print(f"{v:>12} -> 32 bit: {v & 0xFFFFFFFF:032b} -> khôi phục: {to_signed32(v & 0xFFFFFFFF)}")

           5 -> 32 bit: 00000000000000000000000000000101 -> khôi phục: 5
          -5 -> 32 bit: 11111111111111111111111111111011 -> khôi phục: -5
          -1 -> 32 bit: 11111111111111111111111111111111 -> khôi phục: -1
 -2147483648 -> 32 bit: 10000000000000000000000000000000 -> khôi phục: -2147483648
  2147483647 -> 32 bit: 01111111111111111111111111111111 -> khôi phục: 2147483647


In [10]:
def single_bit_count(nums: List[int]) -> int:
    result = 0
    for b in range(32):
        count = sum((x >> b) & 1 for x in nums)
        if count % 3:
            result |= 1 << b
    return to_signed32(result)

print(single_bit_count([2, 2, 3, 2]), single_bit_count([0, 1, 0, 1, 0, 1, 99]), single_bit_count([-2, -2, 1, 1, -3, 1, -3, -3, -4, -2]))

3 99 -4


`(x >> b) & 1` với `x` âm trong Python vẫn cho đúng bit `b` của biểu diễn bù hai (vì Python mô phỏng bù hai với vô hạn bit 1 phía trước). Chỉ cần chuyển kết quả cuối về có dấu.

Độ phức tạp: `32 · n` = `O(n)`; bộ nhớ `O(1)`.

# 6. Mô phỏng

In [11]:
def trace_bits(nums, bits=8):
    result = 0
    print("nums:", nums)
    for x in nums:
        print(f"  {x:>4} = {x & ((1 << bits) - 1):0{bits}b}")
    for b in range(bits):
        count = sum((x >> b) & 1 for x in nums)
        bit = count % 3
        result |= bit << b
        print(f"bit {b}: đếm = {count}, % 3 = {bit}")
    print("đáp án =", f"{result:0{bits}b}", "=", result)
    return result

_ = trace_bits([2, 2, 3, 2], bits=4)

nums: [2, 2, 3, 2]
     2 = 0010
     2 = 0010
     3 = 0011
     2 = 0010
bit 0: đếm = 1, % 3 = 1
bit 1: đếm = 4, % 3 = 1
bit 2: đếm = 0, % 3 = 0
bit 3: đếm = 0, % 3 = 0
đáp án = 0011 = 3


# 7. Cách 3: máy trạng thái `ones` / `twos` — một lượt

Với **mỗi bit**, ta cần đếm 0 → 1 → 2 → 0 (modulo 3). Biểu diễn trạng thái bằng hai bit `(twos, ones)`: `00 → 01 → 10 → 00`. Làm **song song cho cả 32 bit** bằng phép toán bit trên hai số nguyên:

```text
ones = (ones ^ x) & ~twos
twos = (twos ^ x) & ~ones
```

Sau khi duyệt hết, bit nào xuất hiện `3k + 1` lần nằm ở trạng thái `01` → `ones` là đáp án.

In [12]:
def single_state_machine(nums: List[int]) -> int:
    ones = twos = 0
    for x in nums:
        ones = (ones ^ x) & ~twos
        twos = (twos ^ x) & ~ones
    return ones

print(single_state_machine([2, 2, 3, 2]), single_state_machine([0, 1, 0, 1, 0, 1, 99]), single_state_machine([-2, -2, 1, 1, -3, 1, -3, -3, -4, -2]))

3 99 -4


Vì sao đúng cho từng bit (xét một bit, `x` là 0 hoặc 1)? Bảng chuyển trạng thái `(twos, ones)` khi gặp `x = 1`:

| trước | `ones` mới = `(ones ^ 1) & ~twos` | `twos` mới = `(twos ^ 1) & ~ones_mới` | sau |
|---|---|---|---|
| `00` | `1 & 1 = 1` | `1 & 0 = 0` | `01` |
| `01` | `0 & 1 = 0` | `1 & 1 = 1` | `10` |
| `10` | `1 & 0 = 0` | `0 & 1 = 0` | `00` |

Khi `x = 0`: `ones = ones & ~twos`, `twos = twos & ~ones` — trạng thái hợp lệ (`00`, `01`, `10`) giữ nguyên. Python xử lý số âm đúng ở đây vì `~`, `&`, `^` đều hoạt động theo bù hai vô hạn bit — không cần chuyển đổi.

In [13]:
def trace_state(nums, bits=4):
    ones = twos = 0
    print(f"{'x':>4} {'x (bin)':>8} {'ones':>6} {'twos':>6}")
    for x in nums:
        ones = (ones ^ x) & ~twos
        twos = (twos ^ x) & ~ones
        m = (1 << bits) - 1
        print(f"{x:>4} {x & m:>8b} {ones & m:>6b} {twos & m:>6b}")
    print("ones =", ones)
    return ones

_ = trace_state([2, 2, 3, 2])

   x  x (bin)   ones   twos
   2       10     10      0
   2       10      0     10
   3       11      1      0
   2       10     11      0
ones = 3


# 8. Phiên bản đúng format LeetCode

In [14]:
class Solution:
    def singleNumber(self, nums: List[int]) -> int:
        ones = twos = 0
        for x in nums:
            ones = (ones ^ x) & ~twos
            twos = (twos ^ x) & ~ones
        return ones

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<14} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.singleNumber([2, 2, 3, 2]), 3)
check("Ví dụ 2", sol.singleNumber([0, 1, 0, 1, 0, 1, 99]), 99)
check("Một phần tử", sol.singleNumber([7]), 7)
check("Số âm", sol.singleNumber([-2, -2, 1, 1, -3, 1, -3, -3, -4, -2]), -4)
check("INT_MIN", sol.singleNumber([-2**31, 5, 5, 5]), -2**31)
check("INT_MAX", sol.singleNumber([2**31 - 1, -1, -1, -1]), 2**31 - 1)

random.seed(137)
for _ in range(5000):
    k = random.randint(0, 6)
    vals = random.sample(range(-2**31, 2**31), k + 1)
    single, triples = vals[0], vals[1:]
    nums = [single] + triples * 3
    random.shuffle(nums)
    assert sol.singleNumber(nums) == single == single_bit_count(nums) == single_counter(nums) == single_math(nums)
print("5000 mảng ngẫu nhiên (toàn dải 32-bit): 4 cách khớp nhau")

Ví dụ 1        PASS  3
Ví dụ 2        PASS  99
Một phần tử    PASS  7
Số âm          PASS  -4
INT_MIN        PASS  -2147483648
INT_MAX        PASS  2147483647


5000 mảng ngẫu nhiên (toàn dải 32-bit): 4 cách khớp nhau


# 9. Vì sao đúng? Độ phức tạp?

**Đếm bit:** với mỗi bit `b`, `count_b = 3·(số phần tử lặp có bit b) + bit_b(đáp án)` ⇒ `count_b mod 3 = bit_b(đáp án)`.

**Máy trạng thái:** mỗi bit của `(twos, ones)` là một bộ đếm modulo 3 riêng (bảng ở mục 7). Sau `n` phần tử, bit đếm được `3k` lần ở trạng thái `00`, bit đếm được `3k+1` lần ở `01`.

```text
Đếm bit:        Time O(32·n), Space O(1)
Máy trạng thái: Time O(n),    Space O(1)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Dùng XOR như #136

Như mục 4.

## Lỗi 2: Đếm bit nhưng quên chuyển về số có dấu

In [16]:
def wrong_no_sign(nums):
    result = 0
    for b in range(32):
        if sum((x >> b) & 1 for x in nums) % 3:
            result |= 1 << b
    return result                                  # SAI: thiếu to_signed32

print("Sai :", wrong_no_sign([-4, 1, 1, 1]), "| Đúng:", sol.singleNumber([-4, 1, 1, 1]))

Sai : 4294967292 | Đúng: -4


`4294967292` là biểu diễn **không dấu** 32-bit của `-4`.

## Lỗi 3: Đếm bit đến 64 hoặc "đến khi hết bit" với số âm

Với số âm, `x >> b` không bao giờ thành 0 trong Python → vòng `while x` chạy mãi. Phải cố định 32 bit.

In [17]:
def wrong_while_shift(x, limit=100):
    steps = 0
    while x and steps < limit:                     # SAI với x âm: -1 >> 1 == -1
        x >>= 1
        steps += 1
    return steps

print("Số bước dịch phải 5 tới 0:", wrong_while_shift(5), "| với -5 (bị chặn ở 100):", wrong_while_shift(-5))

Số bước dịch phải 5 tới 0: 3 | với -5 (bị chặn ở 100): 100


## Lỗi 4: Cập nhật `twos` trước `ones` trong máy trạng thái

Thứ tự hai dòng quan trọng: `twos` dùng `ones` **mới**. Đảo thứ tự sẽ thay đổi bảng chuyển trạng thái.

In [18]:
def wrong_order(nums):
    ones = twos = 0
    for x in nums:
        twos = (twos ^ x) & ~ones                  # SAI: đảo thứ tự
        ones = (ones ^ x) & ~twos
    return ones

print("Sai :", wrong_order([2, 2, 3, 2]), "| Đúng:", sol.singleNumber([2, 2, 3, 2]))

Sai : 0 | Đúng: 3


(Bản đảo thứ tự thực chất là một máy trạng thái khác: phần tử duy nhất nằm ở `twos` thay vì `ones`.)

# 11. Mẹo nhận diện pattern

**"Mọi phần tử xuất hiện k lần, trừ một"** → đếm bit modulo `k` (tổng quát cho mọi `k`); với `k = 2` rút gọn thành XOR.

Bài liên quan: **#136 Single Number** (k = 2), **#260 Single Number III** (hai số xuất hiện một lần — XOR rồi tách theo một bit khác nhau), **#268 Missing Number**, **#191 Number of 1 Bits**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Đếm bit cho `[5, 5, 5, 6]` (`101, 101, 101, 110`) và tìm đáp án.

## Bài 2 — Điền chỗ trống (đếm bit)

In [19]:
def exercise_2_single_number(nums: List[int]) -> int:
    result = 0
    for b in range(32):
        # TODO: đếm số phần tử có bit b bật; nếu % 3 thì bật bit b của result
        pass
    return to_signed32(result)


# assert exercise_2_single_number([2, 2, 3, 2]) == 3
# assert exercise_2_single_number([0, 1, 0, 1, 0, 1, 99]) == 99
# assert exercise_2_single_number([-4, 1, 1, 1]) == -4

## Bài 3 — Tổng quát: mọi phần tử xuất hiện `k` lần, trừ một phần tử xuất hiện `p` lần (`p % k != 0`)

```python
single_k([4, 4, 4, 4, 9, 9], 4, 2) -> 9
single_k([1, 1, 7, 1], 3, 1)       -> 7
```

In [20]:
def exercise_3_single_k(nums: List[int], k: int, p: int) -> int:
    # TODO
    pass


# assert exercise_3_single_k([4, 4, 4, 4, 9, 9], 4, 2) == 9
# assert exercise_3_single_k([1, 1, 7, 1], 3, 1) == 7
# assert exercise_3_single_k([-3, -3, 2, 2, 2, 2, 2], 5, 2) == -3

## Bài 4 — Biến thể: Single Number III (LeetCode #260)

Mọi phần tử xuất hiện **hai** lần, trừ **hai** phần tử xuất hiện một lần. Tìm hai phần tử đó (thứ tự tùy ý), `O(1)` bộ nhớ.

```python
single_iii([1, 2, 1, 3, 2, 5]) -> [3, 5]
```

In [21]:
def exercise_4_single_iii(nums: List[int]) -> List[int]:
    # TODO
    pass


# assert sorted(exercise_4_single_iii([1, 2, 1, 3, 2, 5])) == [3, 5]
# assert sorted(exercise_4_single_iii([-1, 0])) == [-1, 0]
# assert sorted(exercise_4_single_iii([0, 1])) == [0, 1]

## Bài 5 — Mini challenge: máy trạng thái cho `k = 5`

Viết bản "máy trạng thái" (không đếm 32 bit riêng) cho trường hợp mọi phần tử xuất hiện **5** lần trừ một phần tử xuất hiện **1** lần. Dùng 3 biến bit `a, b, c` làm bộ đếm modulo 5 song song cho mọi bit.

Gợi ý nhanh: coi `(c, b, a)` là một bộ đếm nhị phân 3 bit **cho mỗi vị trí bit**; mỗi khi `x` có bit 1 ở vị trí đó thì cộng 1 (cộng có nhớ bằng `&` và `^`), và khi bộ đếm chạm 5 (`101`) thì xóa về 0 bằng một mặt nạ.

In [22]:
def exercise_5_single_five(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_5_single_five([3, 3, 3, 3, 3, 8]) == 8
# assert exercise_5_single_five([-7, 2, 2, 2, 2, 2]) == -7

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_bits([5, 5, 5, 6], bits=4)` sau khi tự làm.

**Bài 3**
- Mức 1: bit `b` của đáp án bật ⇔ `count_b % k != 0` (vì `count_b % k` là `0` hoặc `p % k`).

**Bài 4**
- Mức 1: `s = XOR toàn bộ = a ^ b ≠ 0`.
- Mức 2: lấy một bit khác nhau: `low = s & -s`; chia mảng theo bit đó và XOR từng nhóm.

**Bài 5**
- Mức 1: cộng 1 vào bộ đếm 3 bit `(c, b, a)` tại các vị trí bit của `x`: `carry_a = a & x; a ^= x; carry_b = b & carry_a; b ^= carry_a; c ^= carry_b`.
- Mức 2: mặt nạ những bit đang bằng 5 (`c & ~b & a`); xóa chúng ở `a` và `c`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [23]:
_ = trace_bits([5, 5, 5, 6], bits=4)

nums: [5, 5, 5, 6]
     5 = 0101
     5 = 0101
     5 = 0101
     6 = 0110
bit 0: đếm = 3, % 3 = 0
bit 1: đếm = 1, % 3 = 1
bit 2: đếm = 4, % 3 = 1
bit 3: đếm = 0, % 3 = 0
đáp án = 0110 = 6


## Đáp án Bài 2

In [24]:
def answer_2_single_number(nums: List[int]) -> int:
    return single_bit_count(nums)

assert answer_2_single_number([2, 2, 3, 2]) == 3
assert answer_2_single_number([0, 1, 0, 1, 0, 1, 99]) == 99
assert answer_2_single_number([-4, 1, 1, 1]) == -4
print("Bài 2 PASS (giống single_bit_count ở mục 5)")

Bài 2 PASS (giống single_bit_count ở mục 5)


## Đáp án Bài 3

In [25]:
def answer_3_single_k(nums: List[int], k: int, p: int) -> int:
    result = 0
    for b in range(32):
        if sum((x >> b) & 1 for x in nums) % k:
            result |= 1 << b
    return to_signed32(result)

assert answer_3_single_k([4, 4, 4, 4, 9, 9], 4, 2) == 9
assert answer_3_single_k([1, 1, 7, 1], 3, 1) == 7
for _ in range(2000):
    k = random.randint(2, 6); p = random.choice([q for q in range(1, 10) if q % k])
    vals = random.sample(range(-1000, 1000), random.randint(1, 5))
    nums = [vals[0]] * p + vals[1:] * k
    random.shuffle(nums)
    assert answer_3_single_k(nums, k, p) == vals[0]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [26]:
def answer_4_single_iii(nums: List[int]) -> List[int]:
    s = reduce(xor, nums)
    low = s & -s
    a = b = 0
    for x in nums:
        if x & low:
            a ^= x
        else:
            b ^= x
    return [a, b]

assert sorted(answer_4_single_iii([1, 2, 1, 3, 2, 5])) == [3, 5]
assert sorted(answer_4_single_iii([-1, 0])) == [-1, 0]
assert sorted(answer_4_single_iii([0, 1])) == [0, 1]
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [27]:
def answer_5_single_five(nums: List[int]) -> int:
    a = b = c = 0                      # bộ đếm 3 bit (c b a) cho mỗi vị trí bit
    for x in nums:
        carry_a = a & x
        a ^= x
        carry_b = b & carry_a
        b ^= carry_a
        c ^= carry_b
        five = c & ~b & a              # vị trí đang đếm tới 5 (101)
        a &= ~five
        c &= ~five
    return a                           # đếm 1 lần -> 001 -> bit a

assert answer_5_single_five([3, 3, 3, 3, 3, 8]) == 8
assert answer_5_single_five([-7, 2, 2, 2, 2, 2]) == -7
for _ in range(2000):
    vals = random.sample(range(-10**6, 10**6), random.randint(1, 5))
    nums = [vals[0]] + vals[1:] * 5
    random.shuffle(nums)
    assert answer_5_single_five(nums) == vals[0]
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> Mỗi bit của đáp án = (số phần tử có bit đó bật) `mod 3`. Đếm 32 bit rồi chuyển về số có dấu, hoặc dùng máy trạng thái `ones = (ones ^ x) & ~twos; twos = (twos ^ x) & ~ones`, trả `ones`. `O(n)` thời gian, `O(1)` bộ nhớ.

---

## Checklist trước khi nộp

- [ ] Không dùng XOR trực tiếp như #136?
- [ ] Đếm bit: cố định 32 bit và chuyển về số có dấu?
- [ ] Máy trạng thái: cập nhật `ones` **trước** `twos`?
- [ ] Kiểm tra với số âm, `INT_MIN`?